In [ ]:
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error 

# ── Load raw data ─────────────────────────────────────────────────────────────
df = pd.read_csv("../data/IPL.csv", low_memory=False)

# ── First innings only ────────────────────────────────────────────────────────
df = df[df['innings'] == 1].copy()

# ── Standardize team names ────────────────────────────────────────────────────
df['batting_team'] = df['batting_team'].replace({
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru',
    'Rising Pune Supergiant': 'Rising Pune Supergiants'
})
df['bowling_team'] = df['bowling_team'].replace({
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru',
    'Rising Pune Supergiant': 'Rising Pune Supergiants'
})

old_teams = ['Deccan Chargers','Kochi Tuskers Kerala','Pune Warriors',
             'Gujarat Lions','Rising Pune Supergiants']
df = df[~df['batting_team'].isin(old_teams)]
df = df[~df['bowling_team'].isin(old_teams)] 

# ── Standardize venue names ───────────────────────────────────────────────────
df['venue'] = df['venue'].str.split(',').str[0]
df['venue'] = df['venue'].replace({
    'M.Chinnaswamy Stadium':                              'M Chinnaswamy Stadium',
    'Punjab Cricket Association IS Bindra Stadium':       'Punjab Cricket Association Stadium',
    'Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium':'ACA-VDCA Cricket Stadium',
    'Feroz Shah Kotla':                                   'Arun Jaitley Stadium'
})

# ── Compute final score per match ─────────────────────────────────────────────
final_scores = df.groupby('match_id')['team_runs'].max().reset_index()
final_scores.rename(columns={'team_runs': 'final_score'}, inplace=True)
df = df.merge(final_scores, on='match_id', how='left')

# ── Core features ─────────────────────────────────────────────────────────────
df['overs_completed']  = df['over'] + df['ball'] / 6
df['current_score']    = df['team_runs']
df['wickets_lost']     = df['team_wicket']
df['current_run_rate'] = df['current_score'] / df['overs_completed'].replace(0, np.nan)
df['current_run_rate'] = df['current_run_rate'].fillna(0)

# Keep only end of over rows
df = df[df['ball'] == 6].copy()
df = df.reset_index(drop=True)

# ── Year and venue average (last 3 seasons) ───────────────────────────────────
df['date'] = pd.to_datetime(df['date'], dayfirst=True)
df['year'] = df['date'].dt.year

# Get one row per match (the final score only)
# Correct venue average calculation:
# Step 1: Load raw data fresh (before any filtering)
raw = pd.read_csv("../data/IPL.csv", low_memory=False)

# Step 2: First innings only
raw = raw[raw['innings'] == 1].copy()

# Step 3: Standardize venue names (same as before)
raw['venue'] = raw['venue'].str.split(',').str[0]
raw['venue'] = raw['venue'].replace({
    'M.Chinnaswamy Stadium':                               'M Chinnaswamy Stadium',
    'Punjab Cricket Association IS Bindra Stadium':        'Punjab Cricket Association Stadium',
    'Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium': 'ACA-VDCA Cricket Stadium',
    'Feroz Shah Kotla':                                    'Arun Jaitley Stadium'
})

# Step 4: Extract year
raw['date'] = pd.to_datetime(raw['date'], dayfirst=True)
raw['year'] = raw['date'].dt.year

# Step 5: For each match, sum ALL ball runs to get true final score
# This is the correct way — add every delivery's runs per match
match_totals = raw.groupby(['match_id', 'venue', 'year'])['runs_total'].sum().reset_index()
match_totals.rename(columns={'runs_total': 'true_final_score'}, inplace=True)

# Step 6: Filter last 3 seasons only
recent_matches = match_totals[match_totals['year'].isin([2023, 2024, 2025])]

# Step 7: Average true final scores per venue
venue_first_avg = recent_matches.groupby('venue')['true_final_score'].mean()

print("✅ Correct Venue Averages (last 3 seasons, true innings totals):")
for v, avg in venue_first_avg.sort_values(ascending=False).items():
    print(f"  {v}: {round(avg, 1)}")

df['venue_first_innings_avg'] = df['venue'].map(venue_first_avg)
df['venue_first_innings_avg'] = df['venue_first_innings_avg'].fillna(
    df['venue_first_innings_avg'].mean()
)

# ── Target encoding ───────────────────────────────────────────────────────────
batting_enc = df.groupby('batting_team')['final_score'].mean()
bowling_enc = df.groupby('bowling_team')['final_score'].mean()
venue_enc   = df.groupby('venue')['final_score'].mean()
global_avg  = float(df['final_score'].mean())

df['batting_team_strength'] = df['batting_team'].map(batting_enc).fillna(global_avg)
df['bowling_team_strength'] = df['bowling_team'].map(bowling_enc).fillna(global_avg)

# ── Quality batting left ──────────────────────────────────────────────────────
# Top 5 batters contribute ~80% of runs. Tail barely scores.
# This is why 100/5 predicts much lower than 100/2 at the same CRR.
position_weights  = [0.18, 0.18, 0.17, 0.15, 0.12, 0.08, 0.05, 0.03, 0.02, 0.02]
quality_remaining = {w: round(sum(position_weights[w:]), 4) for w in range(10)}

print("Quality batting remaining:")
for k, v in quality_remaining.items():
    print(f"  {k} wickets lost → {v*100:.1f}% quality left")

df['quality_batting_left'] = df['wickets_lost'].map(quality_remaining).fillna(0.0)

# ── Game phase ────────────────────────────────────────────────────────────────
def get_phase(overs):
    if overs <= 6:    return 0   # Powerplay
    elif overs <= 15: return 1   # Middle
    else:             return 2   # Death

df['game_phase']      = df['overs_completed'].apply(get_phase)
df['balls_remaining'] = (20 - df['overs_completed']) * 6
df['wickets_in_hand'] = 10 - df['wickets_lost']

# Wicket adjusted projection — unlike naive CRR x overs,
# this correctly shows 100/5 projecting far lower than 100/2
df['wicket_adj_projection'] = (
    df['current_score'] +
    (df['balls_remaining'] / 6) * df['current_run_rate'] * df['quality_batting_left']
)

df['crr_wicket_index'] = df['current_run_rate'] * (df['wickets_in_hand'] / 10)
df['balls_completed']  = df['overs_completed'] * 6
df['balls_per_wicket'] = df['balls_completed'] / (df['wickets_lost'] + 1)
df['projected_score']  = (df['current_score'] / df['balls_completed'].replace(0, np.nan)) * 120
df['projected_score']  = df['projected_score'].fillna(0)
df['runs_to_venue_avg']= df['venue_first_innings_avg'] - df['current_score']
df['wicket_pressure']  = df['wickets_lost'] / (df['overs_completed'] + 0.1)

phase_avg_rr           = {0: 8.5, 1: 8.0, 2: 10.5}
df['phase_avg_rr']     = df['game_phase'].map(phase_avg_rr)
df['rr_vs_phase_norm'] = df['current_run_rate'] - df['phase_avg_rr']

# ── Features ──────────────────────────────────────────────────────────────────
FEATURES = [
    "batting_team_strength",
    "bowling_team_strength",
    "venue_first_innings_avg",
    "overs_completed",
    "current_score",
    "wickets_lost",
    "wickets_in_hand",
    "current_run_rate",
    "crr_wicket_index",
    "balls_per_wicket",
    "balls_remaining",
    "projected_score",
    "runs_to_venue_avg",
    "game_phase",
    "quality_batting_left",
    "wicket_adj_projection",
    "wicket_pressure",
    "rr_vs_phase_norm"
]

# ── Train on last 3 seasons ───────────────────────────────────────────────────
df_train = df[df['year'].isin([2023, 2024, 2025])].copy()
df_train = df_train.dropna(subset=FEATURES + ['final_score'])

X = df_train[FEATURES]
y = df_train['final_score']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

from xgboost import XGBRegressor
print("Training model...")
model = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, n_jobs=-1, random_state=42)
model.fit(X_train, y_train)
print("Done!")

mae = mean_absolute_error(y_test, model.predict(X_test))
print(f"\nModel MAE: {round(mae, 2)} runs")

# ── Active teams (current 10 IPL franchises only) ─────────────────────────────
TEAMS = sorted([
    'Chennai Super Kings',
    'Delhi Capitals',
    'Gujarat Titans',
    'Kolkata Knight Riders',
    'Lucknow Super Giants',
    'Mumbai Indians',
    'Punjab Kings',
    'Rajasthan Royals',
    'Royal Challengers Bengaluru',
    'Sunrisers Hyderabad'
])

# ── Active venues (only venues used in IPL 2023-25) ───────────────────────────
VENUES = sorted([
    'ACA-VDCA Cricket Stadium',
    'Arun Jaitley Stadium',
    'Barsapara Cricket Stadium',
    'Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium',
    'Dr DY Patil Sports Academy',
    'Eden Gardens',
    'Himachal Pradesh Cricket Association Stadium',
    'M Chinnaswamy Stadium',
    'MA Chidambaram Stadium',
    'Maharaja Yadavindra Singh International Cricket Stadium',
    'Maharashtra Cricket Association Stadium',
    'Narendra Modi Stadium',
    'Punjab Cricket Association Stadium',
    'Rajiv Gandhi International Stadium',
    'Sawai Mansingh Stadium',
    'Wankhede Stadium'
])

# ── Save all 4 pickle files ───────────────────────────────────────────────────
pickle.dump(model,             open("ipl_score_model.pkl",    "wb"))
pickle.dump(FEATURES,          open("model_columns.pkl",      "wb"))
pickle.dump(quality_remaining, open("quality_remaining.pkl",  "wb"))
pickle.dump({
    "teams":            TEAMS,
    "venues":           VENUES,
    "batting_strength": batting_enc,
    "bowling_strength": bowling_enc,
    "venue_avg":        venue_enc,
    "global_avg":       global_avg,
}, open("ipl_encoders.pkl", "wb"))

print("\n✅ All files saved:")
print("   ipl_score_model.pkl")
print("   model_columns.pkl")
print("   quality_remaining.pkl")
print("   ipl_encoders.pkl")
print(f"\nTeams : {len(TEAMS)}")
print(f"Venues: {len(VENUES)}")
print(f"Features: {len(FEATURES)}")

✅ Correct Venue Averages (last 3 seasons, true innings totals):
  ACA-VDCA Cricket Stadium: 208.8
  Arun Jaitley Stadium: 201.5
  Punjab Cricket Association Stadium: 197.8
  Eden Gardens: 196.7
  Narendra Modi Stadium: 195.3
  Himachal Pradesh Cricket Association Stadium: 194.3
  Rajiv Gandhi International Stadium: 189.4
  M Chinnaswamy Stadium: 189.1
  Sawai Mansingh Stadium: 187.2
  Wankhede Stadium: 187.0
  Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium: 175.4
  Barsapara Cricket Stadium: 174.6
  Maharaja Yadavindra Singh International Cricket Stadium: 168.9
  MA Chidambaram Stadium: 168.3
Quality batting remaining:
  0 wickets lost → 100.0% quality left
  1 wickets lost → 82.0% quality left
  2 wickets lost → 64.0% quality left
  3 wickets lost → 47.0% quality left
  4 wickets lost → 32.0% quality left
  5 wickets lost → 20.0% quality left
  6 wickets lost → 12.0% quality left
  7 wickets lost → 7.0% quality left
  8 wickets lost → 4.0% quality left
  9 wickets lost →